In [1]:
# Google Colab setup: fetch repository and set working directory
from pathlib import Path
import os

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists():
    !git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming

NOTEBOOK_DIR = REPO_ROOT / 'module_2/week_6/use_case_3'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')

Cloning into '/content/BITS_programming'...
remote: Enumerating objects: 2101, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 2101 (delta 11), reused 15 (delta 3), pack-reused 2066 (from 1)
Receiving objects: 100% (2101/2101), 263.06 MiB | 11.66 MiB/s, done.
Resolving deltas: 100% (400/400), done.
Updating files: 100% (1348/1348), done.
Working directory: /content/BITS_programming/module_2/week_6/use_case_3


In [2]:
# AWS credentials setup via Google Colab Secrets
import os

def get_colab_secret(name, required=True):
    try:
        from google.colab import userdata
        value = os.environ.get(name) or userdata.get(name)
    except Exception as exc:
        if required:
            raise RuntimeError(f"Unable to read Colab Secret: {name}") from exc
        return None
    if required and (value is None or value == ""):
        raise RuntimeError(f"Add the Colab Secret {name} and grant this notebook access.")
    return value

AWS_ACCESS_KEY_ID = get_colab_secret("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = get_colab_secret("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = get_colab_secret("AWS_SESSION_TOKEN", required=False)

os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
if AWS_SESSION_TOKEN:
    os.environ["AWS_SESSION_TOKEN"] = AWS_SESSION_TOKEN

_aws_region = os.getenv("AWS_REGION") or os.getenv("AWS_DEFAULT_REGION") or "ap-south-1"
os.environ["AWS_REGION"] = _aws_region
os.environ["AWS_DEFAULT_REGION"] = _aws_region

print(f"AWS credentials loaded. Region: {_aws_region}")

AWS credentials loaded. Region: ap-south-1


In [3]:
# Install boto3 and create globally unique S3 buckets using AWS Account ID
!pip install boto3 -q
import os
import boto3
from botocore.exceptions import ClientError

region = os.environ.get("AWS_REGION", "ap-south-1")

sts_client = boto3.client("sts", region_name=region)
account_id = sts_client.get_caller_identity()["Account"]

_s3 = boto3.client("s3", region_name=region)

BASE_BUCKET_NAMES = ['usecase-etl-1', 'usecase-etl-2']
REQUIRED_BUCKETS = [f"{b}-{account_id}" for b in BASE_BUCKET_NAMES]

for _b in REQUIRED_BUCKETS:
    try:
        if region == "us-east-1":
            _s3.create_bucket(Bucket=_b)
        else:
            _s3.create_bucket(
                Bucket=_b,
                CreateBucketConfiguration={"LocationConstraint": region}
            )
        print(f"Created bucket: {_b}")
    except ClientError as _e:
        _code = _e.response.get("Error", {}).get("Code", "")
        if _code in ("BucketAlreadyExists", "BucketAlreadyOwnedByYou", "Conflict"):
            print(f"Bucket already exists (reusing existing): {_b}")
        else:
            print(f"Could not create bucket {_b} ({_code}): {_e}")

print("Buckets ready:", REQUIRED_BUCKETS)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 7.4 MB/s eta 0:00:00
Bucket already exists (reusing existing): usecase-etl-1-455865672536
Bucket already exists (reusing existing): usecase-etl-2-455865672536
Buckets ready: ['usecase-etl-1-455865672536', 'usecase-etl-2-455865672536']


In [6]:
import json, os, pandas as pd, numpy as np
from pathlib import Path
from io import BytesIO, StringIO
import boto3

region = os.environ.get("AWS_REGION", "ap-south-1")
sts_client = boto3.client("sts", region_name=region)
account_id = sts_client.get_caller_identity()["Account"]

BUCKET_1 = f"usecase-etl-1-{account_id}"
BUCKET_2 = f"usecase-etl-2-{account_id}"

# S3 URIs
RAW_S3_URI = f"s3://{BUCKET_1}/churn/raw/telco_customer_churn_sample.csv"
TRAIN_S3_URI = f"s3://{BUCKET_2}/ml_ready/train.csv"
TEST_S3_URI = f"s3://{BUCKET_2}/ml_ready/test.csv"
VALIDATION_S3_URI = f"s3://{BUCKET_2}/artifacts/validation_summary_generated.csv"

# Dynamic local path search across repo
REPO_ROOT = Path('/content/BITS_programming')
possible_raw_paths = [
    Path("../04_Datasets/raw/telco_customer_churn_sample.csv"),
    REPO_ROOT / "04_Datasets" / "raw" / "telco_customer_churn_sample.csv",
    REPO_ROOT / "module_2" / "week_6" / "use_case_3" / "04_Datasets" / "raw" / "telco_customer_churn_sample.csv",
] + list(REPO_ROOT.glob("**/telco_customer_churn_sample.csv"))

LOCAL_RAW_PATH = next((p for p in possible_raw_paths if p.exists()), possible_raw_paths[0])
LOCAL_TRAIN_PATH = Path("../04_Datasets/ml_ready/train.csv")
LOCAL_TEST_PATH = Path("../04_Datasets/ml_ready/test.csv")
LOCAL_VALIDATION_PATH = Path("../05_Artifacts/validation_summary_generated.csv")

s3_client = boto3.client("s3", region_name=region)

def parse_s3_uri(uri: str):
    bucket, key = uri.replace('s3://', '', 1).split('/', 1)
    return bucket, key

# Auto-seed raw dataset into S3 if missing
def seed_raw_data_to_s3():
    b_raw, k_raw = parse_s3_uri(RAW_S3_URI)
    try:
        s3_client.head_object(Bucket=b_raw, Key=k_raw)
        print("ℹ️ Raw data already present in S3 bucket.")
    except Exception:
        if LOCAL_RAW_PATH.exists():
            print(f"📦 Seeding raw data from local ({LOCAL_RAW_PATH}) to S3 ({RAW_S3_URI})...")
            s3_client.upload_file(str(LOCAL_RAW_PATH), b_raw, k_raw)
            print("✅ Seed completed successfully.")
        else:
            print(f"⚠️ Local raw dataset file not found at {LOCAL_RAW_PATH}")

seed_raw_data_to_s3()

def read_csv_aws_first(s3_uri: str, local_path: Path) -> pd.DataFrame:
    try:
        bucket, key = parse_s3_uri(s3_uri)
        obj = s3_client.get_object(Bucket=bucket, Key=key)
        print("✅ Reading from S3:", s3_uri)
        return pd.read_csv(BytesIO(obj['Body'].read()))
    except Exception as e:
        print(f"⚠️ S3 read failed ({e}). Falling back to local:", local_path)
        return pd.read_csv(local_path)

def write_csv_aws_first(df: pd.DataFrame, s3_uri: str, local_path: Path) -> None:
    csv_buffer = StringIO()
    df.to_csv(csv_buffer, index=False)
    try:
        bucket, key = parse_s3_uri(s3_uri)
        s3_client.put_object(Bucket=bucket, Key=key, Body=csv_buffer.getvalue().encode('utf-8'))
        print("✅ Written to S3:", s3_uri)
    except Exception as e:
        print(f"⚠️ S3 write failed ({e}). Writing locally:", local_path)
        local_path.parent.mkdir(parents=True, exist_ok=True)
        local_path.write_text(csv_buffer.getvalue(), encoding='utf-8')

print("S3 read/write helpers ready.")

📦 Seeding raw data from local (/content/BITS_programming/module_2/week_6/use_case_3_local/telco_customer_churn_sample.csv) to S3 (s3://usecase-etl-1-455865672536/churn/raw/telco_customer_churn_sample.csv)...
✅ Seed completed successfully.
S3 read/write helpers ready.


# Use Case 3 - ETL for ML Preparation on AWS

This lab treats churn modeling as a **data engineering process** first and a machine-learning workflow second.


## What this lab covers
- Amazon S3 raw and curated zones
- Notebook ETL profiling and transformation
- Optional AWS Glue productionization bridge
- Amazon SageMaker training and Model Registry


## ETL flow in one line
`Extract from S3 raw -> Transform and validate in notebook -> Load curated train/test to S3 -> Train in SageMaker -> Register model version`


## 1. Extract - land raw data
In the live AWS demo, show the same file in `s3://<bucket>/churn/raw/` before reading it here locally.


In [8]:
df_raw = read_csv_aws_first(RAW_S3_URI, LOCAL_RAW_PATH)
df_raw.head()

✅ Reading from S3: s3://usecase-etl-1-455865672536/churn/raw/telco_customer_churn_sample.csv


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-0000,Male,0,Yes,Yes,50,No,No phone service,DSL,No,...,No,Yes,Yes,Yes,Month-to-month,Yes,Mailed check,50.98,2552.87,No
1,7590-0001,Male,0,No,Yes,56,No,No phone service,DSL,No,...,No,No,No,Yes,Month-to-month,No,Electronic check,38.44,2128.40,No
2,7590-0002,Female,0,No,Yes,54,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Bank transfer (automatic),27.31,1458.53,No
3,7590-0003,Male,0,No,Yes,41,No,No phone service,Fiber optic,Yes,...,No,No,No,No,One year,Yes,Bank transfer (automatic),67.71,2805.57,No
4,7590-0004,Male,0,No,Yes,11,No,No phone service,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Electronic check,19.42,161.12,No


## 2. Profile the source before transformation
Profile nulls, blanks, duplicates, and label quality. This is still ETL. The downstream consumer just happens to be a model.


In [9]:
profile = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "null_count": df_raw.isna().sum(),
    "blank_count": df_raw.astype(str).apply(lambda s: s.str.strip().eq("")).sum()
}).sort_values(["null_count", "blank_count"], ascending=False)
profile.head(15)

,dtype,null_count,blank_count
TotalCharges,object,0,24
customerID,object,0,0
gender,object,0,0
SeniorCitizen,int64,0,0
Partner,object,0,0
Dependents,object,0,0
tenure,int64,0,0
PhoneService,object,0,0
MultipleLines,object,0,0
InternetService,object,0,0


In [10]:
print("Duplicate customer IDs:", df_raw["customerID"].duplicated().sum())
print(df_raw["Churn"].value_counts(dropna=False))

Duplicate customer IDs: 0
Churn
No     471
Yes    179
Name: count, dtype: int64


## 3. Transform - standardize fields and engineer features


In [11]:
from sklearn.model_selection import train_test_split

df = df_raw.copy()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].replace(" ", np.nan), errors="coerce")
df["label"] = df["Churn"].map({"Yes": 1, "No": 0})
df["is_new_customer"] = (df["tenure"] <= 6).astype(int)
df["monthly_charge_band"] = pd.cut(
    df["MonthlyCharges"],
    bins=[0, 35, 70, 200],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)
df["avg_monthly_spend_gap"] = df["TotalCharges"] - (df["tenure"] * df["MonthlyCharges"])
feature_df = df.copy()
feature_df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,label,is_new_customer,monthly_charge_band,avg_monthly_spend_gap
0,7590-0000,Male,0,Yes,Yes,50,No,No phone service,DSL,No,...,Month-to-month,Yes,Mailed check,50.98,2552.87,No,0,0,Medium,3.87
1,7590-0001,Male,0,No,Yes,56,No,No phone service,DSL,No,...,Month-to-month,No,Electronic check,38.44,2128.40,No,0,0,Medium,-24.24
2,7590-0002,Female,0,No,Yes,54,Yes,Yes,No,No internet service,...,Month-to-month,No,Bank transfer (automatic),27.31,1458.53,No,0,0,Low,-16.21
3,7590-0003,Male,0,No,Yes,41,No,No phone service,Fiber optic,Yes,...,One year,Yes,Bank transfer (automatic),67.71,2805.57,No,0,0,Medium,29.46
4,7590-0004,Male,0,No,Yes,11,No,No phone service,No,No internet service,...,Two year,No,Electronic check,19.42,161.12,No,0,0,Low,-52.50


## 4. Validate curated data before publish


In [12]:
validation_summary = {
    "row_count": int(feature_df.shape[0]),
    "null_totalcharges": int(feature_df["TotalCharges"].isna().sum()),
    "null_labels": int(feature_df["label"].isna().sum()),
    "duplicate_customer_ids": int(feature_df["customerID"].duplicated().sum())
}
validation_summary

{'row_count': 650,
 'null_totalcharges': 24,
 'null_labels': 0,
 'duplicate_customer_ids': 0}

## 5. Load - publish train/test outputs


In [13]:
train_df, test_df = train_test_split(
    feature_df,
    test_size=0.2,
    random_state=42,
    stratify=feature_df["label"]
)

write_csv_aws_first(train_df, TRAIN_S3_URI, LOCAL_TRAIN_PATH)
write_csv_aws_first(test_df, TEST_S3_URI, LOCAL_TEST_PATH)
write_csv_aws_first(
    pd.DataFrame([validation_summary]),
    VALIDATION_S3_URI,
    LOCAL_VALIDATION_PATH
)

print(f"Train output shape: {train_df.shape} | Test output shape: {test_df.shape}")

✅ Written to S3: s3://usecase-etl-2-455865672536/ml_ready/train.csv
✅ Written to S3: s3://usecase-etl-2-455865672536/ml_ready/test.csv
✅ Written to S3: s3://usecase-etl-2-455865672536/artifacts/validation_summary_generated.csv
Train output shape: (520, 25) | Test output shape: (130, 25)


## 6. Optional AWS Glue bridge
Open `06_Assets/code/optional_glue_job_uc3.py` to explain how the notebook transforms can become a repeatable Glue PySpark job.


## 7. Train and register
In the live demo, use SageMaker to train and then explain Model Registry as the governed publish point for the model artifact.


In [14]:
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))

Current Time in IST: 2026-09-14 11:01:32
